<a href="https://colab.research.google.com/github/natchanant-arch/Data-ess/blob/main/semi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ธุรกิจสหกรณ์ออมทรัพย์ (Savings Cooperative)

โดยมีระบบบัญชี ฝาก-ถอน-โอน และสมาชิกจะสามารถทำธุรกรรมได้ทีละรายการ (ฝาก/ถอน/โอน) มีการตรวจสอบยอดเงินเพียงพอก่อนทำรายการ คำนวณดอกเบี้ยจากเงินคงเหลือในบัญชีในช่วงสิ้นปี

## ส่วนที่ 0 — Import เพื่อ เรียกใช้งานชุดคำสั่ง หรือฟังก์ชันสำเร็จรูป

In [ ]:
import random
import time
from datetime import datetime, timedelta
!pip install Faker
from faker import Faker
fake = Faker("th_TH")
random.seed(1)
import pandas as pd
import matplotlib, os, shutil
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

In [ ]:
# 1. ติดตั้งฟอนต์ภาษาไทย
!apt-get -y install fonts-thai-tlwg

# 2. ล้าง cache ของ matplotlib เพื่ออัปเดตฟอนต์ใหม่
cache_dir = matplotlib.get_cachedir()
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# 3. ลงทะเบียนฟอนต์ใหม่เข้ากับ FontManager
font_path = '/usr/share/fonts/truetype/tlwg/Loma.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)

# 4. ตั้งค่าฟอนต์หลัก
plt.rcParams['font.family'] = 'Loma'
plt.rcParams['axes.unicode_minus'] = False

print("ตั้งค่าระบบฟอนต์ภาษาไทยสำเร็จ!")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-thai-tlwg is already the newest version (1:0.7.3-1).
0 upgraded, 0 newly installed, 0 to remove and 4 not upgraded.
ตั้งค่าระบบฟอนต์ภาษาไทยสำเร็จ!


## ส่วนที่ 1 — เตรียม class และฟังก์ชัน

In [ ]:
class Member:
    """ข้อมูลสมาชิกธนาคารออมทรัพย์"""
    def __init__(self, member_id, customer_name, citizen_id="-", phone_number="-"):
        self.member_id = member_id
        self.customer_name = customer_name
        self.citizen_id = citizen_id
        self.phone_number = phone_number

    def get_info(self):
        return f"ลูกค้า ID: {self.member_id} | ชื่อ: {self.customer_name} | เลขบัตรประชาชน: {self.citizen_id} | เบอร์โทร: {self.phone_number}"

In [ ]:
class Account:
    """บัญชีเงินฝากธนาคารออมทรัพย์"""
    def __init__(self, account_number, balance, owner, interest_rate=0.015):
        self.account_number = account_number
        self.balance = float(balance)
        self.owner = owner #ชื่อเจ้าของบัญชี
        self.interest_rate = interest_rate  # ดอกเบี้ย 1.5% ต่อปี ( default )

    # Validation & Operations การตรวจสอบเงื่อนไขละการดำเนินงาน
    def deposit(self, amount):
        """ฝากเงิน: balance = balance + amount"""
        self.balance += amount
        return "ฝากเงินสำเร็จ"

    def withdraw(self, amount):
        """ถอนเงิน: ตรวจสอบ balance >= amount"""
        if self.balance < amount:
            return f"ยอดเงินไม่พอ (มีอยู่ {self.balance:,.2f} บาท)"
        self.balance -= amount
        return "ถอนเงินสำเร็จ"

    def transfer(self, target_account, amount):
        """โอนเงิน: ตัดบัญชีต้นทาง และบวกเข้าบัญชีปลายทาง"""
        if self.balance < amount:
            return f" ยอดเงินไม่พอโอน (มีอยู่ {self.balance:,.2f} บาท)"
        self.balance -= amount
        target_account.balance += amount
        return "โอนเงินสำเร็จ"

    def apply_interest(self):
        """คำนวณดอกเบี้ย: interest = balance * interest_rate แล้วบวกเข้ายอดคงเหลือ"""
        interest = self.balance * self.interest_rate
        self.balance += interest
        return interest



---


มีการแปลงชนิดข้อมูลของยอดเงินให้เป็น Float (จำนวนจริง/ทศนิยม)
และบันทึกเข้าตัวแปรของวัตถุบัญชี เพื่อให้รองรับการคำนวณเศษสตางค์และนำไปบวกลบเงินได้แม่นยำ

---



In [ ]:
import random
from datetime import datetime, timedelta

# 1. ฟังก์ชันสุ่มคิว 40 รายการต่อวัน (รับแค่ txn_id)
def generate_transaction_data(txn_id):
    base_date = datetime(2026, 8, 22)
    day_idx = 0
    total_items = 0

    while True:
        items_today = 40  # กำหนด 40 รายการต่อวันคงที่

        if txn_id <= total_items + items_today:
            queue_num = txn_id - total_items
            current_date = base_date + timedelta(days=day_idx)

            return {
                "วันที่": current_date.strftime("%d/%m/%Y"),
                "หมายเลขคิว": f"A-{queue_num:03d}",
                "queue_seq": queue_num - 1
            }

        total_items += items_today
        day_idx += 1

# 2. Class Transaction
class Transaction:
    def __init__(self, txn_id, account, transaction_type, amount, target_account=None):
        self.txn_id = txn_id

        # 📌 เรียกใช้โดยส่งค่า 40 ตามลำดับตำแหน่งตรงๆ
        date_info = generate_transaction_data(txn_id)
        self.queue_number = date_info["หมายเลขคิว"]
        self.txn_date = date_info["วันที่"]

        # ⏰ คำนวณเวลาเดินหน้าตามลำดับคิวในวันนั้น โดยให้นาทีที่จะบวกเพิ่มเข้าไปจากเวลาเริ่มต้น
        base_start_time = datetime.strptime("08:30:00", "%H:%M:%S")
        queue_seq = date_info["queue_seq"]

        minutes_added = queue_seq * random.randint(8, 11) + random.randint(0, 2) #ลำดับคิวมาคูณกับเวลาสุ่มต่อคิว 8 - 11 นาที และบวกเพิ่มเศษเวลาสุ่มอีกเล็กน้อย 0- 2 นาที
        seconds_added = random.randint(0, 59) # สุ่มเวลาในช่วงวินาทีตั้งแต่ 0 - 59

        actual_time = base_start_time + timedelta(minutes=minutes_added, seconds=seconds_added)
        self.time = actual_time.strftime("%H:%M:%S")

        self.account = account
        self.account_number = account.account_number
        self.customer_name = account.owner.customer_name
        self.transaction_type = transaction_type
        self.amount = amount
        self.target_account = target_account

    def to_dict(self):
        interest = getattr(self.account, "yearly_interest", 0.0)

        if isinstance(self.customer_name, (tuple, list)):
            fname, lname = self.customer_name[0], self.customer_name[1]
        else:
            parts = str(self.customer_name).split(" ", 1)
            fname = parts[0]
            lname = parts[1] if len(parts) > 1 else "-"

        return {
            "ID รายการ": self.txn_id,
            "หมายเลขคิว": self.queue_number,
            "วันที่ทำรายการ": self.txn_date,
            "เวลาทำรายการ": self.time,
            "เลขบัญชี": self.account_number,
            "ชื่อ": fname,
            "นามสกุล": lname,
            "ประเภทรายการ": self.transaction_type,
            "จำนวนเงิน": self.amount,
            "บัญชีปลายทาง": self.target_account if self.target_account else "-",
            "ยอดหลังทำรายการ": round(self.account.balance, 2),
            "ดอกเบี้ยสิ้นปี (1.5%)": round(interest, 2),
            "ยอดรวมดอกเบี้ยสุทธิ": round(self.account.balance + interest, 2)
        }

เป็นการรันวันที่แบบอัตโนมัติ โดยใช้ timedelta เพิ่มจำนวนวันไปเรื่อยๆ เมื่อคิวการทำธุรกรรมในวันนั้นรันจนครบ 40 รายการ

In [ ]:
def generate_thai_name():
    """ฟังก์ชัน: สุ่มชื่อและนามสกุลลูกค้าแยกกัน"""
    name = fake.name()
    first_name, last_name = name.split(" ", 1)

    return f"{first_name} {last_name}"

def random_amount(min_val=100.0, max_val=2000.0):
    """ฟังก์ชัน: สุ่มยอดเงิน -> คืนค่าเป็น float """
    return round(random.uniform(min_val, max_val), 2)

def format_currency(amount, symbol="บาท"):
    """ฟังก์ชัน: จัดรูปแบบตัวเลขเป็นสตริงราคา -> คืนค่าเป็น string"""
    return f"{amount:,.2f} {symbol}"

## ส่วนที่ 2 — ทดสอบฟังก์ชันทีละตัว ก่อนเอาไปประกอบเป็นกระบวนการ

In [ ]:
# เรียก generate_thai_name() 3 ครั้ง -> ทุกครั้งได้ชื่อสุ่มไม่ซ้ำแบบ (แสดงว่าฟังก์ชันทำงานทุกครั้งที่เรียก)
for _ in range(3):
    print("ชื่อที่สุ่มได้:", generate_thai_name())

ชื่อที่สุ่มได้: วรรณนิสา นักสำหรวจ
ชื่อที่สุ่มได้: พัชรพร เลิศกิ่ง
ชื่อที่สุ่มได้: ปุณณรัตน์ วุฑฒยากร


In [ ]:
print("\n# เรียก random_amount() 3 ครั้ง")
for _ in range(3):
    print("ยอดเงินสุ่มได้:", format_currency(random_amount()))


# เรียก random_amount() 3 ครั้ง
ยอดเงินสุ่มได้: 355.29 บาท
ยอดเงินสุ่มได้: 1,710.12 บาท
ยอดเงินสุ่มได้: 1,551.17 บาท


## ส่วนที่ 3 — ฟังก์ชันอธิบายขั้นตอนคำนวณราคา (ให้เห็นว่าฟังก์ชันเรียกฟังก์ชัน/method อื่นต่อได้)

In [ ]:
def explain_transaction_calculation(transaction):
    """ฟังก์ชันคำนวณเงิน ฝาก/ถอน/โอน และเรียกใช้ Method ของ Account"""

    account = transaction.account
    amount = float(transaction.amount)
    txn_type = transaction.transaction_type

    print(f"หมายเลขคิว = '{transaction.queue_number}'")
    print(f"หมายเลขบัญชี = '{account.account_number}'")
    print(f"ชื่อลูกค้า = '{transaction.customer_name}'")
    print(f"ประเภทรายการ = '{txn_type}'")
    print(f"ยอดเงินก่อนทำรายการ = {format_currency(account.balance)}")

    # 📌 เรียกใช้ Method ภายใน Class Account
    if txn_type == "ฝากเงิน":
      status_msg = account.deposit(amount)

    elif txn_type == "ถอนเงิน":
        status_msg = account.withdraw(amount)
        # ❌ ถ้าเงินไม่พอ ให้หยุดประมวลผลทันที
        if "ยอดเงินไม่พอ" in status_msg:
          print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
          print(f"ยอดเงินคงเหลือหลังทำรายการ = {format_currency(account.balance)}")
          print(f"สถานะรายการ = '{status_msg}'")
          print("❌ ทำรายการไม่สำเร็จ!")
          return False

    elif txn_type == "โอนเงิน":
        if transaction.target_account:
            print(f"บัญชีปลายทาง = '{transaction.target_account}'")

        dummy_member = Member(0, "บัญชีปลายทาง")
        dummy_target = Account("987-6-00000-0", balance=0.0, owner=dummy_member)
        status_msg = account.transfer(dummy_target, amount)
        # ❌ ถ้าเงินไม่พอโอน ให้หยุดประมวลผลทันที
        if "ยอดเงินไม่พอ" in status_msg:
            print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
            print(f"ยอดเงินคงเหลือหลังทำรายการ = {format_currency(account.balance)}")
            print(f"สถานะรายการ = '{status_msg}'")
            print("❌ ทำรายการไม่สำเร็จ!")
            return False

    # คำนวณดอกเบี้ย (จะทำเฉพาะรายการที่สำเร็จเท่านั้น)
    interest_val = account.apply_interest()

    print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
    print(f"ยอดเงินคงเหลือหลังทำรายการ = {format_currency(account.balance)}")
    print(f"สถานะรายการ = '{status_msg}'")
    print(f"ดอกเบี้ยที่ได้รับเมื่อสิ้นปี (1.5%) = {format_currency(interest_val)}")

    return True

In [ ]:
import random

# 📌 สั่ง Seed ทั้ง Python random และตัวแปร fake คู่กัน
random.seed(1)
fake.seed_instance(1)  # 👈 ใช้ fake.seed_instance(1) เพื่อล็อกค่าตัวแปร fake โดยตรง

transactions = []

# 2. Loop สุ่มข้อมูล 300 รายการ
for i in range(1, 301):
    name = generate_thai_name()
    amount = random_amount()
    service = random.choice(["ฝากเงิน", "ถอนเงิน", "โอนเงิน"])

    member = Member(member_id=1 + i, customer_name=name)

    initial_balance = round(random.uniform(100, 3000), 2)

    # 🎲 สุ่มเลขบัญชีลูกค้า
    acc_p1 = random.randint(100, 999)
    acc_p2 = random.randint(1, 9)
    acc_p3 = random.randint(10000, 99999)
    random_account_no = f"{acc_p1}-{acc_p2}-{acc_p3:05d}-0"

    account = Account(account_number=random_account_no, balance=initial_balance, owner=member)
    target_acc = f"987-6-{random.randint(10000, 99999)}-0" if service == "โอนเงิน" else None
    # 📌 คำนวณยอดเงินผ่าน Method ของ Account โดยตรง
    if service == "ฝากเงิน":
        account.deposit(amount)
    elif service == "ถอนเงิน":
        account.withdraw(amount)
    elif service == "โอนเงิน":
        dummy_mem = Member(0, "ปลายทาง")
        dummy_acc = Account("987-6-00000-0", balance=0.0, owner=dummy_mem)
        account.transfer(dummy_acc, amount)

    # 📌 [เพิ่มบรรทัดนี้] คำนวณเลขคิวให้รีเซ็ตทุกๆ 40 คิว
    daily_queue = ((i - 1) % 40) + 1
    queue_no = f"A-{daily_queue:03d}"

    # ประมวลผลดอกเบี้ย
    account.apply_interest()

    transaction = Transaction(
        txn_id=i,
        account=account,
        transaction_type=service,
        amount=amount,
        target_account=target_acc
    )

    transactions.append(transaction)

# 3. แสดงตัวอย่างรายการแรก (คิว A-001)
print("\n--- [ตัวอย่างการแสดงผลรายการแรก (คิว A-001)] ---")
explain_transaction_calculation(transactions[0])


--- [ตัวอย่างการแสดงผลรายการแรก (คิว A-001)] ---
หมายเลขคิว = 'A-001'
หมายเลขบัญชี = '607-8-71898-0'
ชื่อลูกค้า = 'ชิดชนก เยาวธนโชค'
ประเภทรายการ = 'ฝากเงิน'
ยอดเงินก่อนทำรายการ = 1,212.91 บาท
จำนวนเงินทำรายการ = 355.29 บาท
ยอดเงินคงเหลือหลังทำรายการ = 1,591.73 บาท
สถานะรายการ = 'ฝากเงินสำเร็จ'
ดอกเบี้ยที่ได้รับเมื่อสิ้นปี (1.5%) = 23.52 บาท


True

เป็นการล็อกค่าของการสุ่ม (Random Seed) ไว้ เพื่อให้ทุกครั้งที่กดรันโปรแกรม ระบบจะสุ่มได้ตัวเลขและข้อมูลชุดเดิมเสมอ ทำให้ง่ายต่อการทดสอบและตรวจสอบความถูกต้องของระบบ

## ส่วนที่ 4 — จำลอง "ลูกค้า 1 คนเดินเข้าร้าน" แบบ step-by-step

In [ ]:
import random
import time

def simulate_customer_visit(txn_id, customer_name, pause=0.5):
    """จำลองขั้นตอนลูกค้า 1 คนเดินเข้าธนาคาร"""
    print("=" * 60)
    queue_no = f"A-{txn_id:03d}"
    print(f"🎫 [ผู้ออกบัตรคิว] คุณ '{customer_name}' กดรับบัตรคิว ได้หมายเลข: {queue_no}")

    # 1. สร้าง Member
    member = Member(member_id=100 + txn_id, customer_name=customer_name)

    # 2. 🎲 สุ่มยอดเงินตั้งต้น (100 - 3,000 บาท)
    initial_balance = round(random.uniform(100, 3000), 2)

    # 3. 🎲 สุ่มเลขบัญชีลูกค้า (รูปแบบ 123-4-56789-0)
    acc_p1 = random.randint(100, 999)
    acc_p2 = random.randint(1, 9)
    acc_p3 = random.randint(10000, 99999)
    random_account_no = f"{acc_p1}-{acc_p2}-{acc_p3:05d}-0"

    # 4. สร้าง Account
    account = Account(account_number=random_account_no, balance=initial_balance, owner=member)

    # 5. สุ่มประเภทรายการ และจำนวนเงิน
    service = random.choice(["ฝากเงิน", "ถอนเงิน", "โอนเงิน"])
    amount = random_amount()
    target_acc = f"987-6-{random.randint(10000, 99999)}-0" if service == "โอนเงิน" else None

    # 6. ประมวลผลธุรกรรม
    print(f"🔔 [เชิญหมายเลข {queue_no}] เข้าเคาน์เตอร์บริการ -> แจ้งทำรายการ: '{service}'")
    print("⚙️ เจ้าหน้าที่บันทึกข้อมูลเข้าระบบ (สถานะ: กำลังดำเนินการ)")

    if service == "ฝากเงิน":
        account.deposit(amount)
    elif service == "ถอนเงิน":
        account.withdraw(amount)
    elif service == "โอนเงิน":
        dummy_mem = Member(0, "ปลายทาง")
        dummy_acc = Account("987-6-00000-0", balance=0.0, owner=dummy_mem)
        account.transfer(dummy_acc, amount)

    # คิดดอกเบี้ย
    account.apply_interest()

    # 7. สร้าง Transaction
    txn = Transaction(
        txn_id=txn_id,
        account=account,
        transaction_type=service,
        amount=amount,
        target_account=target_acc
    )

    # แสดงรายละเอียดคำนวณ
    print("🖥️ เจ้าหน้าที่ตรวจสอบยอดเงินและประเภทรายการ:")
    is_success = explain_transaction_calculation(txn)

    # ปรับรูปแบบชื่อกรณีที่เป็น Tuple/List
    display_name = " ".join(customer_name) if isinstance(customer_name, (tuple, list)) else customer_name

    # Check เงื่อนไขพิมพ์สลิป (ลบ print ข้อความไม่สำเร็จใน else ออกเพื่อไม่ให้ซ้ำ)
    if is_success:
        print("✅ ทำรายการสำเร็จ!")
        print(f"🧾 สลิปบันทึกรายการ #{txn.txn_id}: คิว {txn.queue_number} | วันที่ {txn.txn_date} | เวลา {txn.time} | คุณ {display_name} | {service} | ยอด {format_currency(amount)} | ยอดคงเหลือสุทธิ {format_currency(account.balance)}")
    else:
        # ไม่ต้องใส่ print("❌ ทำรายการไม่สำเร็จ!") ซ้ำตรงนี้แล้ว
        print(f"🧾 สลิปบันทึกรายการ #{txn.txn_id}: คิว {txn.queue_number} | วันที่ {txn.txn_date} | เวลา {txn.time} | คุณ {display_name} | [รายการยกเลิก - ยอดเงินไม่พอ]")

    return txn

In [ ]:
# --- [ทดสอบเรียกใช้งานจริงกับลูกค้า 1 คน] ---
transaction_a = simulate_customer_visit(txn_id=1, customer_name="สมหญิง สายทอง", pause=0.5)

🎫 [ผู้ออกบัตรคิว] คุณ 'สมหญิง สายทอง' กดรับบัตรคิว ได้หมายเลข: A-001
🔔 [เชิญหมายเลข A-001] เข้าเคาน์เตอร์บริการ -> แจ้งทำรายการ: 'โอนเงิน'
⚙️ เจ้าหน้าที่บันทึกข้อมูลเข้าระบบ (สถานะ: กำลังดำเนินการ)
🖥️ เจ้าหน้าที่ตรวจสอบยอดเงินและประเภทรายการ:
หมายเลขคิว = 'A-001'
หมายเลขบัญชี = '524-6-86672-0'
ชื่อลูกค้า = 'สมหญิง สายทอง'
ประเภทรายการ = 'โอนเงิน'
ยอดเงินก่อนทำรายการ = 1,853.35 บาท
บัญชีปลายทาง = '987-6-69568-0'
จำนวนเงินทำรายการ = 189.87 บาท
ยอดเงินคงเหลือหลังทำรายการ = 1,688.43 บาท
สถานะรายการ = 'โอนเงินสำเร็จ'
ดอกเบี้ยที่ได้รับเมื่อสิ้นปี (1.5%) = 24.95 บาท
✅ ทำรายการสำเร็จ!
🧾 สลิปบันทึกรายการ #1: คิว A-001 | วันที่ 22/08/2026 | เวลา 08:32:58 | คุณ สมหญิง สายทอง | โอนเงิน | ยอด 189.87 บาท | ยอดคงเหลือสุทธิ 1,688.43 บาท


##  ส่วนที่ 5 — จำลองลูกค้าหลายคนเดินเข้าธนาคารต่อเนื่องกัน


In [ ]:
walk_in_customers = []

for i in range(2, 11):
    customer_name = fake.name()

    walk_in_customers.append(
        Member(
            member_id=i,
            customer_name=customer_name
        )
    )
completed_transactions = []  # เก็บผลลัพธ์ของทุกรายการในรอบนี้

for i, cust in enumerate(walk_in_customers, start=2):
    # ส่งชื่อลูกค้าเข้าฟังก์ชัน simulate_customer_visit
    txn = simulate_customer_visit(txn_id=i, customer_name=cust.customer_name, pause=0.3)
    completed_transactions.append(txn)

print("=" * 60)
print(f"🏁 จบการสาธิต — วันนี้มีลูกค้าเข้าทำรายการทั้งหมด {len(completed_transactions) + 1} คน (รวมคิว #1 ก่อนหน้า)")

🎫 [ผู้ออกบัตรคิว] คุณ 'ดวงพร หนุนสุข' กดรับบัตรคิว ได้หมายเลข: A-002
🔔 [เชิญหมายเลข A-002] เข้าเคาน์เตอร์บริการ -> แจ้งทำรายการ: 'ฝากเงิน'
⚙️ เจ้าหน้าที่บันทึกข้อมูลเข้าระบบ (สถานะ: กำลังดำเนินการ)
🖥️ เจ้าหน้าที่ตรวจสอบยอดเงินและประเภทรายการ:
หมายเลขคิว = 'A-002'
หมายเลขบัญชี = '698-1-55790-0'
ชื่อลูกค้า = 'ดวงพร หนุนสุข'
ประเภทรายการ = 'ฝากเงิน'
ยอดเงินก่อนทำรายการ = 3,365.16 บาท
จำนวนเงินทำรายการ = 1,698.26 บาท
ยอดเงินคงเหลือหลังทำรายการ = 5,139.37 บาท
สถานะรายการ = 'ฝากเงินสำเร็จ'
ดอกเบี้ยที่ได้รับเมื่อสิ้นปี (1.5%) = 75.95 บาท
✅ ทำรายการสำเร็จ!
🧾 สลิปบันทึกรายการ #2: คิว A-002 | วันที่ 22/08/2026 | เวลา 08:41:40 | คุณ ดวงพร หนุนสุข | ฝากเงิน | ยอด 1,698.26 บาท | ยอดคงเหลือสุทธิ 5,139.37 บาท
🎫 [ผู้ออกบัตรคิว] คุณ 'อมัด ธูปะวิโรจน์' กดรับบัตรคิว ได้หมายเลข: A-003
🔔 [เชิญหมายเลข A-003] เข้าเคาน์เตอร์บริการ -> แจ้งทำรายการ: 'ฝากเงิน'
⚙️ เจ้าหน้าที่บันทึกข้อมูลเข้าระบบ (สถานะ: กำลังดำเนินการ)
🖥️ เจ้าหน้าที่ตรวจสอบยอดเงินและประเภทรายการ:
หมายเลขคิว = 'A-003'
หมายเลขบัญชี = '252-1-57896-0

## ส่วนที่ 6 — สรุปผล (ยืนยันว่าฟังก์ชันคืนค่าถูกต้องและใช้ต่อได้จริง

In [ ]:
import pandas as pd

summary_rows = []
random.seed(1)

for t in transactions:
    cust_name = " ".join(t.account.owner.customer_name) if isinstance(t.account.owner.customer_name, (tuple, list)) else t.account.owner.customer_name

    # 📌 วิธีเช็กสถานะที่ถูกต้อง:
    # ถ้ารายการเป็น ถอน/โอน แล้วยอดคงเหลือเหลือน้อยกว่ายอดที่ขอถอน/โอน แสดงว่าทำรายการไม่สำเร็จ
    if t.transaction_type in ["ถอนเงิน", "โอนเงิน"] and t.account.balance < t.amount:
        status_str = "ไม่สำเร็จ (ยอดเงินไม่พอ)"
    else:
        status_str = "สำเร็จ"

    summary_rows.append({
        "txn_id": t.txn_id,
        "queue_number": t.queue_number,  # ดึงคิว A-001 ถึง A-040 ที่ถูกคำนวณมาใช้ได้เลย
        "date": t.txn_date,              # ดึงวันที่เปลี่ยนตามวันจริงมาใช้ได้เลย
        "customer_name": cust_name,
        "account_number": t.account.account_number,
        "transaction_type": t.transaction_type,
        "amount": round(t.amount, 2),
        "balance_after": round(t.account.balance, 2),
        "status": status_str,
        "target_account": t.target_account if t.target_account else "-"
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,txn_id,queue_number,date,customer_name,account_number,transaction_type,amount,balance_after,status,target_account
0,1,A-001,22/08/2026,ชิดชนก เยาวธนโชค,607-8-71898-0,ฝากเงิน,355.29,1591.73,สำเร็จ,-
1,2,A-002,22/08/2026,พุทธิพงษ์ ดัตพันธุ์,880-1-68377-0,ถอนเงิน,1026.93,333.01,ไม่สำเร็จ (ยอดเงินไม่พอ),-
2,3,A-003,22/08/2026,เพ็ญยุภา แท่นทอง,131-1-13335-0,ฝากเงิน,534.65,3297.52,สำเร็จ,-
3,4,A-004,22/08/2026,นวรรษนันท์ ทวีเดช,640-4-67394-0,ถอนเงิน,511.54,1718.83,สำเร็จ,-
4,5,A-005,22/08/2026,ฌาฆีภัตฐ์ ธรรมนิยม,570-5-12816-0,โอนเงิน,756.83,745.49,ไม่สำเร็จ (ยอดเงินไม่พอ),987-6-64549-0
...,...,...,...,...,...,...,...,...,...,...
295,296,A-016,29/08/2026,อณัฐตา บุนยะศัพท์,932-2-62465-0,ฝากเงิน,1086.05,3550.65,สำเร็จ,-
296,297,A-017,29/08/2026,มนัญชยา ถนอมพล,767-8-97432-0,โอนเงิน,1465.40,152.77,ไม่สำเร็จ (ยอดเงินไม่พอ),987-6-46782-0
297,298,A-018,29/08/2026,ฉลองชัย น้ำทิพย์,476-9-36993-0,ถอนเงิน,1331.30,1312.42,ไม่สำเร็จ (ยอดเงินไม่พอ),-
298,299,A-019,29/08/2026,ศศิรินทร์ อุลหัสสา,503-8-90825-0,ฝากเงิน,552.20,1371.47,สำเร็จ,-


In [ ]:
# Save to csv
summary_df.to_csv("สหกรณ์ออมทรัพย์.csv")

### ตารางลูกค้า

In [ ]:
import pandas as pd

summary_rows = []

for t in transactions:
    cust_name = " ".join(t.account.owner.customer_name) if isinstance(t.account.owner.customer_name, (tuple, list)) else t.account.owner.customer_name

    # 📌 วิธีเช็กสถานะที่ถูกต้อง:
    # ถ้ารายการเป็น ถอน/โอน แล้วยอดคงเหลือเหลือน้อยกว่ายอดที่ขอถอน/โอน แสดงว่าทำรายการไม่สำเร็จ
    if t.transaction_type in ["ถอนเงิน", "โอนเงิน"] and t.account.balance < t.amount:
        status_str = "ไม่สำเร็จ (ยอดเงินไม่พอ)"
    else:
        status_str = "สำเร็จ"

    summary_rows.append({
        "txn_id": t.txn_id,
        "queue_number": t.queue_number,  # ดึงคิว A-001 ถึง A-040 ที่ถูกคำนวณมาใช้ได้เลย
        "date": t.txn_date,              # ดึงวันที่เปลี่ยนตามวันจริงมาใช้ได้เลย
        "customer_name": cust_name,
        "account_number": t.account.account_number,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,txn_id,queue_number,date,customer_name,account_number
0,1,A-001,22/08/2026,ชิดชนก เยาวธนโชค,607-8-71898-0
1,2,A-002,22/08/2026,พุทธิพงษ์ ดัตพันธุ์,880-1-68377-0
2,3,A-003,22/08/2026,เพ็ญยุภา แท่นทอง,131-1-13335-0
3,4,A-004,22/08/2026,นวรรษนันท์ ทวีเดช,640-4-67394-0
4,5,A-005,22/08/2026,ฌาฆีภัตฐ์ ธรรมนิยม,570-5-12816-0
...,...,...,...,...,...
295,296,A-016,29/08/2026,อณัฐตา บุนยะศัพท์,932-2-62465-0
296,297,A-017,29/08/2026,มนัญชยา ถนอมพล,767-8-97432-0
297,298,A-018,29/08/2026,ฉลองชัย น้ำทิพย์,476-9-36993-0
298,299,A-019,29/08/2026,ศศิรินทร์ อุลหัสสา,503-8-90825-0


In [ ]:
# Save to csv
summary_df.to_csv("ลูกค้าสหกรณ์ออมทรัพย์.csv")

### ตารางธุรกรรม

In [ ]:
import pandas as pd

summary_rows = []

for t in transactions:
    cust_name = " ".join(t.account.owner.customer_name) if isinstance(t.account.owner.customer_name, (tuple, list)) else t.account.owner.customer_name

    # 📌 วิธีเช็กสถานะที่ถูกต้อง:
    # ถ้ารายการเป็น ถอน/โอน แล้วยอดคงเหลือเหลือน้อยกว่ายอดที่ขอถอน/โอน แสดงว่าทำรายการไม่สำเร็จ
    if t.transaction_type in ["ถอนเงิน", "โอนเงิน"] and t.account.balance < t.amount:
        status_str = "ไม่สำเร็จ (ยอดเงินไม่พอ)"
    else:
        status_str = "สำเร็จ"

    summary_rows.append({
        "queue_number": t.queue_number,
        "transaction_type": t.transaction_type,
        "amount": round(t.amount, 2),
        "balance_after": round(t.account.balance, 2),
        "status": status_str,
        "target_account": t.target_account if t.target_account else "-"
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,queue_number,transaction_type,amount,balance_after,status,target_account
0,A-001,ฝากเงิน,355.29,1591.73,สำเร็จ,-
1,A-002,ถอนเงิน,1026.93,333.01,ไม่สำเร็จ (ยอดเงินไม่พอ),-
2,A-003,ฝากเงิน,534.65,3297.52,สำเร็จ,-
3,A-004,ถอนเงิน,511.54,1718.83,สำเร็จ,-
4,A-005,โอนเงิน,756.83,745.49,ไม่สำเร็จ (ยอดเงินไม่พอ),987-6-64549-0
...,...,...,...,...,...,...
295,A-016,ฝากเงิน,1086.05,3550.65,สำเร็จ,-
296,A-017,โอนเงิน,1465.40,152.77,ไม่สำเร็จ (ยอดเงินไม่พอ),987-6-46782-0
297,A-018,ถอนเงิน,1331.30,1312.42,ไม่สำเร็จ (ยอดเงินไม่พอ),-
298,A-019,ฝากเงิน,552.20,1371.47,สำเร็จ,-


เป็นการเช็กความถูกต้องของการทำธุรกรรม โดยตรวจสอบว่าถ้ารายการนั้นเป็นการถอนหรือโอน แต่เงินในบัญชีจริงมีน้อยกว่ายอดที่ต้องการถอน/โอน ระบบจะเปลี่ยนสถานะเป็น "ไม่สำเร็จ" ทันที

In [ ]:
# Save to csv
summary_df.to_csv("ธุรกรรมสหกรณ์ออมทรัพย์.csv")


## ส่วนที่ 7 — สรุปผล 300 คน

In [ ]:
# คำนวณยอดหมุนเวียนรวมทำรายการในรอบ demo
total_volume = sum(t.amount for t in transactions)
print(f"\n📊 ยอดธุรกรรมทำรายการรวมจากลูกค้าที่เดินเข้าธนาคารในรอบ demo นี้: {format_currency(total_volume)}")


📊 ยอดธุรกรรมทำรายการรวมจากลูกค้าที่เดินเข้าธนาคารในรอบ demo นี้: 312,954.47 บาท
